# Importação

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import os
import numpy as np
import time

# ETL

In [ ]:
# tratamento de arquivos censo
from google.colab import drive

start_time = time.time()

# 1. Montar Google Drive
drive.mount('/content/drive')

PASTA_DRIVE = '/content/drive/MyDrive/dados_enem/censo/'

# 2. Dicionário de De/Para (Colunas Originais do INEP -> Seus Nomes Finais)
MAPEAMENTO_COLUNAS = {
    'CO_ENTIDADE': 'CO_ESCOLA',
    'CO_MUNICIPIO': 'COD_MUNICIPIO_IBGE',
    'SG_UF': 'UF',
    'NO_REGIAO': 'REGIAO',
    'NO_ENTIDADE': 'NOME_ESCOLA',
    'TP_DEPENDENCIA': 'REDE_ENSINO',
    'TP_SITUACAO_FUNCIONAMENTO': 'SITUACAO_FUNCIONAMENTO',
    'IN_INTERNET': 'TEM_INTERNET_ESCOLA',
    'IN_INTERNET_ALUNOS': 'INTERNET_DISPONIVEL_ALUNOS',
    'IN_INTERNET_APRENDIZAGEM': 'INTERNET_USO_PEDAGOGICO',
    'IN_BANDA_LARGA': 'TEM_BANDA_LARGA',
    'IN_LABORATORIO_INFORMATICA': 'TEM_LAB_INFORMATICA',
    'QT_DESKTOP_ALUNO': 'QT_COMPUTADORES_DESKTOP',
    'QT_COMP_PORTATIL_ALUNO': 'QT_NOTEBOOKS_ALUNO',
    'QT_TABLET_ALUNO': 'QT_TABLETS_ALUNO',
    'NU_ANO_CENSO': 'ANO_CENSO'
}

colunas_inep = list(MAPEAMENTO_COLUNAS.keys())

# 3. Arquivos apenas de 2024 e 2025
arquivos_csv = {
    2024: PASTA_DRIVE + 'microdados_ed_basica_2024.csv',
    2025: PASTA_DRIVE + 'microdados_ed_basica_2025.csv'
}

dataframes_tratados = []

print("🚀 Iniciando o processamento dos Censos 2024 e 2025...\n")

for ano, caminho in arquivos_csv.items():
    print(f"⏳ Processando Censo {ano}...")
    
    # Leitura otimizada
    try:
        df = pd.read_csv(caminho, sep=';', encoding='latin1', low_memory=False)
    except Exception:
        df = pd.read_csv(caminho, sep=',', encoding='utf-8', low_memory=False)
    
    # Garantir a presença da coluna do ano
    df['NU_ANO_CENSO'] = ano
    
    # Manter apenas as colunas mapeadas que existem no arquivo
    cols_existentes = [c for c in colunas_inep if c in df.columns]
    df_sub = df[cols_existentes].copy()
    
    # Tratamento dos Indicadores Binários (IN_): 0 = Não, 1 = Sim, 2 = Não informado
    cols_indicadores = [c for c in cols_existentes if c.startswith('IN_')]
    for col in cols_indicadores:
        df_sub[col] = df_sub[col].fillna(2)
        df_sub[col] = pd.to_numeric(df_sub[col], errors='coerce').fillna(2)
        df_sub[col] = df_sub[col].apply(lambda x: int(x) if x in [0, 1] else 2).astype('uint8')
        
    # Tratamento das Quantidades (QT_): Preencher valores nulos com 0
    cols_quantidades = [c for c in cols_existentes if c.startswith('QT_')]
    for col in cols_quantidades:
        df_sub[col] = pd.to_numeric(df_sub[col], errors='coerce').fillna(0).astype('int32')
        
    # Renomear colunas para a sua nomenclatura final
    df_sub = df_sub.rename(columns=MAPEAMENTO_COLUNAS)
    
    dataframes_tratados.append(df_sub)
    print(f"   ✓ {ano} lido e renomeado: {df_sub.shape[0]:,} linhas | {df_sub.shape[1]} colunas.")

# 4. Empilhamento Vertical (Dados em Painel 2024 - 2025)
print("\n🥞 Empilhando 2024 e 2025...")
df_painel = pd.concat(dataframes_tratados, ignore_index=True)

# 5. Ordenação das colunas na ordem exata solicitada
ordem_desejada = [
    'CO_ESCOLA', 'COD_MUNICIPIO_IBGE', 'UF', 'REGIAO', 'NOME_ESCOLA',
    'REDE_ENSINO', 'SITUACAO_FUNCIONAMENTO', 'TEM_INTERNET_ESCOLA',
    'INTERNET_DISPONIVEL_ALUNOS', 'INTERNET_USO_PEDAGOGICO',
    'TEM_BANDA_LARGA', 'TEM_LAB_INFORMATICA', 'QT_COMPUTADORES_DESKTOP',
    'QT_NOTEBOOKS_ALUNO', 'QT_TABLETS_ALUNO', 'ANO_CENSO'
]

cols_finais = [c for c in ordem_desejada if c in df_painel.columns]
df_painel = df_painel[cols_finais]

# 6. Salvar base consolidada no Drive
caminho_saida = PASTA_DRIVE + 'microdados_censo_2024_2025_tecnologia.csv'
print(f"\n💾 Salvando em: {caminho_saida}")
df_painel.to_csv(caminho_saida, index=False, sep=';', encoding='utf-8-sig')

tempo_total = time.time() - start_time
print(f"\n✅ CONCLUÍDO EM {tempo_total:.2f} SEGUNDOS!")
print("=" * 60)
print(f"📊 Total de linhas empilhadas: {df_painel.shape[0]:,}")
print(f"📋 Total de colunas finais: {df_painel.shape[1]}")
print("=" * 60)

# 7. Exibir visualização das primeiras linhas
display(df_painel.head(5))

In [ ]:
# tratamento de arquivos censo
# Dicionários de mapeamento
MAP_REDE_ENSINO = {
    1: 'Federal',
    2: 'Estadual',
    3: 'Municipal',
    4: 'Privada'
}

MAP_SITUACAO_FUNCIONAMENTO = {
    1: 'Em Atividade',
    2: 'Paralisada',
    3: 'Extinta (ano do Censo)',
    4: 'Extinta em Anos Anteriores'
}

#Converter para numérico e aplicar o mapeamento
df['REDE_ENSINO'] = pd.to_numeric(df['REDE_ENSINO'], errors='coerce').map(MAP_REDE_ENSINO).fillna('Não Informado')
df['SITUACAO_FUNCIONAMENTO'] = pd.to_numeric(df['SITUACAO_FUNCIONAMENTO'], errors='coerce').map(MAP_SITUACAO_FUNCIONAMENTO).fillna('Não Informado')

#Sobrescrever o CSV corrigido no seu Google Drive
caminho_saida = '/content/drive/MyDrive/dados_enem/censo/microdados_censo_2024_2025_tecnologia.csv'
df.to_csv(caminho_saida, index=False, sep=';', encoding='utf-8-sig')

print("✅ Colunas REDE_ENSINO e SITUACAO_FUNCIONAMENTO convertidas com sucesso!")

#Visualizar as 10 primeiras linhas atualizadas
display(df.head(10))

In [ ]:
df_enem_2 = pd.read_csv('./data/raw/PARTICIPANTES_2024.csv',sep=';',encoding='ISO-8859-1')
df_enem_2.head()

,NU_INSCRICAO,NU_ANO,TP_FAIXA_ETARIA,TP_SEXO,TP_ESTADO_CIVIL,TP_COR_RACA,TP_NACIONALIDADE,TP_ST_CONCLUSAO,TP_ANO_CONCLUIU,TP_ENSINO,...,Q014,Q015,Q016,Q017,Q018,Q019,Q020,Q021,Q022,Q023
0,210062064233,2024,5,F,1,1,1,1,3,NaN,...,A,B,B,B,D,A,B,B,E,A
1,210062064234,2024,11,F,1,1,1,1,10,NaN,...,A,A,A,A,B,A,B,A,C,A
2,210062064235,2024,11,F,1,1,1,1,9,NaN,...,A,B,B,B,D,B,B,A,D,A
3,210062064236,2024,3,F,1,3,1,2,0,1.0,...,A,B,A,A,A,A,B,A,D,A
4,210062064237,2024,16,M,3,1,1,1,18,NaN,...,A,A,A,A,B,A,B,A,B,A


In [ ]:
df_enem_2.columns

Index(['NU_INSCRICAO', 'NU_ANO', 'TP_FAIXA_ETARIA', 'TP_SEXO',
       'TP_ESTADO_CIVIL', 'TP_COR_RACA', 'TP_NACIONALIDADE', 'TP_ST_CONCLUSAO',
       'TP_ANO_CONCLUIU', 'TP_ENSINO', 'IN_TREINEIRO', 'CO_MUNICIPIO_PROVA',
       'NO_MUNICIPIO_PROVA', 'CO_UF_PROVA', 'SG_UF_PROVA', 'Q001', 'Q002',
       'Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011',
       'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 'Q019', 'Q020',
       'Q021', 'Q022', 'Q023'],
      dtype='str')

In [ ]:
df_enem_3 = pd.read_csv('./data/raw/PARTICIPANTES_2025.csv',sep=';',encoding='ISO-8859-1')
#df_enem_3.columns
#df_enem_3.info()
df_enem_3.head()

,NU_INSCRICAO,NU_ANO,TP_FAIXA_ETARIA,TP_SEXO,TP_ESTADO_CIVIL,TP_COR_RACA,TP_NACIONALIDADE,TP_ST_CONCLUSAO,TP_ANO_CONCLUIU,TP_ENSINO,...,Q014,Q015,Q016,Q017,Q018,Q019,Q020,Q021,Q022,Q023
0,210066506229,2025,6,F,1,2,2,1,4,NaN,...,A,A,A,A,A,A,A,A,B,A
1,210066506230,2025,3,M,1,1,1,2,0,1.0,...,A,B,B,A,C,A,B,B,B,A
2,210066506231,2025,2,F,1,1,1,3,0,NaN,...,A,A,B,A,A,A,B,C,C,C
3,210066506232,2025,2,M,1,3,2,2,0,1.0,...,A,B,A,A,B,A,B,B,D,D
4,210066506233,2025,1,F,1,1,1,3,0,NaN,...,B,B,B,B,D,B,B,D,E,D


In [ ]:
df_resultados_2025 = pd.read_csv('./data/raw/RESULTADOS_2025.csv',sep=';',encoding='ISO-8859-1')
#df_resultados_2025[['NU_SEQUENCIAL']]
df_resultados_2025.columns
#df_resultados_2025.info()
#df_resultados_2025.head()

Index(['NU_SEQUENCIAL', 'NU_ANO', 'CO_ESCOLA', 'CO_MUNICIPIO_ESC',
       'NO_MUNICIPIO_ESC', 'CO_UF_ESC', 'SG_UF_ESC', 'TP_DEPENDENCIA_ADM_ESC',
       'TP_LOCALIZACAO_ESC', 'TP_SIT_FUNC_ESC', 'CO_MUNICIPIO_PROVA',
       'NO_MUNICIPIO_PROVA', 'CO_UF_PROVA', 'SG_UF_PROVA', 'TP_PRESENCA_CN',
       'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT', 'CO_PROVA_CN',
       'CO_PROVA_CH', 'CO_PROVA_LC', 'CO_PROVA_MT', 'NU_NOTA_CN', 'NU_NOTA_CH',
       'NU_NOTA_LC', 'NU_NOTA_MT', 'TX_RESPOSTAS_CN', 'TX_RESPOSTAS_CH',
       'TX_RESPOSTAS_LC', 'TX_RESPOSTAS_MT', 'TP_LINGUA', 'TX_GABARITO_CN',
       'TX_GABARITO_CH', 'TX_GABARITO_LC', 'TX_GABARITO_MT',
       'TP_STATUS_REDACAO', 'NU_NOTA_COMP1', 'NU_NOTA_COMP2', 'NU_NOTA_COMP3',
       'NU_NOTA_COMP4', 'NU_NOTA_COMP5', 'NU_NOTA_REDACAO',
       'TP_STATUS_REDACAO_AV1', 'NU_NOTA_AV1', 'NU_NOTA_COMP1_AV1',
       'NU_NOTA_COMP2_AV1', 'NU_NOTA_COMP3_AV1', 'NU_NOTA_COMP4_AV1',
       'NU_NOTA_COMP5_AV1', 'TP_STATUS_REDACAO_AV2', 'NU_NOTA_AV

In [ ]:
df_resultados_2024 = pd.read_csv('./data/raw/RESULTADOS_2024.csv',sep=';',encoding='ISO-8859-1')
df_resultados_2024.head()

,NU_SEQUENCIAL,NU_ANO,CO_ESCOLA,CO_MUNICIPIO_ESC,NO_MUNICIPIO_ESC,CO_UF_ESC,SG_UF_ESC,TP_DEPENDENCIA_ADM_ESC,TP_LOCALIZACAO_ESC,TP_SIT_FUNC_ESC,...,TX_GABARITO_CH,TX_GABARITO_LC,TX_GABARITO_MT,TP_STATUS_REDACAO,NU_NOTA_COMP1,NU_NOTA_COMP2,NU_NOTA_COMP3,NU_NOTA_COMP4,NU_NOTA_COMP5,NU_NOTA_REDACAO
0,206403,2024,23052929.0,2301406.0,Aratuba,23.0,CE,2.0,1.0,1.0,...,CECEBEBCDBADDEBBABCDCAECEDADBAEABEADCEDADACBC,CAAAECDDDAECBECEDDCBDEDDCECBDCBCEADBBDBDDCBEDA...,CECEBEDADCAADECDBBCEBDCCCACABBABBADDDCEADBBCE,1.0,80.0,60.0,60.0,80.0,20.0,300.0
1,3604651,2024,42103770.0,4218004.0,Tijucas,42.0,SC,4.0,1.0,1.0,...,DADABCDCECEDEBBBEBCDBADACBCADCEBEADBAECAECEDA,AACEADDDACDADECBBDBEEBDDECCBEDDCADBBDEBCEBDCEC...,DBCECEACADABBECBEDADCCDCCABBBCECEBCEADABBADDD,1.0,160.0,200.0,200.0,180.0,180.0,920.0
2,1461268,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,CAEDBADBEBCCEDADBAEACBCADCEDEBBEBACECEABCDDAD,EAAACDDCDADDCBDEECEECBDCDADEEBCBEBADBCEBDBDDDD...,ABBABBADDDBBCECEADCEBCCCDBADCBEDAECADACDBCECE,1.0,120.0,120.0,40.0,120.0,80.0,480.0
3,4301058,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,ACBCADCEDBAECEDADBADBEBCDADABCDEABCAEDEBBCECE,AACEADACDDCBEDADEEBDDBDBBCEECBDCBADDDCBDEDDCEC...,CEBCEADBBCEADDDABBABBCCDCCECEDBACADECABEDADCB,1.0,140.0,200.0,160.0,160.0,80.0,740.0
4,3148322,2024,21150354.0,2100436.0,Alto Alegre do Maranhão,21.0,MA,2.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Verificação

In [ ]:
df_participantes = pd.read_csv('./data/processed/PARTICIPANTES_TRATADOS.csv',sep=',', encoding='UTF-8')
df_participantes.head()

C:\Users\hugom\AppData\Local\Temp\ipykernel_26900\1444446181.py:1: DtypeWarning: Columns (0: POSSUI_RENDA) have mixed types. Specify dtype option on import or set low_memory=False.
  df_participantes = pd.read_csv('./data/processed/PARTICIPANTES_TRATADOS.csv',sep=',', encoding='UTF-8')


,NU_INSCRICAO,NU_ANO,TP_FAIXA_ETARIA,TP_SEXO,TP_COR_RACA,CO_MUNICIPIO_PROVA,NO_MUNICIPIO_PROVA,SG_UF_PROVA,POSSUI_RENDA,RENDA_FAMILIAR,TIPO_DE_ESCOLA_EM
0,210062064233,2024,20 anos,F,Branca,4314902,Porto Alegre,RS,True,DE 1 A 1.5 SM,Somente em escola pública.
1,210062064234,2024,Entre 26 e 30 anos,F,Branca,4318903,São Luiz Gonzaga,RS,True,DE 2 A 2.5 SM,Somente em escola pública.
2,210062064235,2024,Entre 26 e 30 anos,F,Branca,4320107,Sarandi,RS,True,DE 6 A 7 SM,Somente em escola pública.
3,210062064236,2024,18 anos,F,Parda,4313409,Novo Hamburgo,RS,False,DE 1 A 1.5 SM,Somente em escola pública.
4,210062064237,2024,Entre 51 e 55 anos,M,Branca,4309209,Gravataí,RS,False,NENHUMA RENDA,Somente em escola pública.


In [ ]:
df_participantes = pd.read_csv('./data/processed/PARTICIPANTES_TRATADOS.csv',sep=',', encoding='UTF-8')
df_participantes.head()

C:\Users\hugom\AppData\Local\Temp\ipykernel_26900\1444446181.py:1: DtypeWarning: Columns (0: POSSUI_RENDA) have mixed types. Specify dtype option on import or set low_memory=False.
  df_participantes = pd.read_csv('./data/processed/PARTICIPANTES_TRATADOS.csv',sep=',', encoding='UTF-8')


,NU_INSCRICAO,NU_ANO,TP_FAIXA_ETARIA,TP_SEXO,TP_COR_RACA,CO_MUNICIPIO_PROVA,NO_MUNICIPIO_PROVA,SG_UF_PROVA,POSSUI_RENDA,RENDA_FAMILIAR,TIPO_DE_ESCOLA_EM
0,210062064233,2024,20 anos,F,Branca,4314902,Porto Alegre,RS,True,DE 1 A 1.5 SM,Somente em escola pública.
1,210062064234,2024,Entre 26 e 30 anos,F,Branca,4318903,São Luiz Gonzaga,RS,True,DE 2 A 2.5 SM,Somente em escola pública.
2,210062064235,2024,Entre 26 e 30 anos,F,Branca,4320107,Sarandi,RS,True,DE 6 A 7 SM,Somente em escola pública.
3,210062064236,2024,18 anos,F,Parda,4313409,Novo Hamburgo,RS,False,DE 1 A 1.5 SM,Somente em escola pública.
4,210062064237,2024,Entre 51 e 55 anos,M,Branca,4309209,Gravataí,RS,False,NENHUMA RENDA,Somente em escola pública.


## Excluindo a coluna 'POSSUI_RENDA' do DF

In [ ]:
#df_participantes = df_participantes.drop(columns=['POSSUI_RENDA'])
df_participantes.to_csv('./data/processed/PARTICIPANTES_LIMPO.csv', index=False, sep=',', encoding='UTF-8')

In [ ]:
df_participantes_limpo = pd.read_csv('./data/processed/PARTICIPANTES_LIMPO.csv',sep=',', encoding='UTF-8')
df_participantes_limpo.head()

,NU_INSCRICAO,NU_ANO,TP_FAIXA_ETARIA,TP_SEXO,TP_COR_RACA,CO_MUNICIPIO_PROVA,NO_MUNICIPIO_PROVA,SG_UF_PROVA,RENDA_FAMILIAR,TIPO_DE_ESCOLA_EM
0,210062064233,2024,20 anos,F,Branca,4314902,Porto Alegre,RS,DE 1 A 1.5 SM,Somente em escola pública.
1,210062064234,2024,Entre 26 e 30 anos,F,Branca,4318903,São Luiz Gonzaga,RS,DE 2 A 2.5 SM,Somente em escola pública.
2,210062064235,2024,Entre 26 e 30 anos,F,Branca,4320107,Sarandi,RS,DE 6 A 7 SM,Somente em escola pública.
3,210062064236,2024,18 anos,F,Parda,4313409,Novo Hamburgo,RS,DE 1 A 1.5 SM,Somente em escola pública.
4,210062064237,2024,Entre 51 e 55 anos,M,Branca,4309209,Gravataí,RS,NENHUMA RENDA,Somente em escola pública.


In [ ]:
df_participantes_limpo.dtypes

NU_INSCRICAO          int64
NU_ANO                int64
TP_FAIXA_ETARIA         str
TP_SEXO                 str
TP_COR_RACA             str
CO_MUNICIPIO_PROVA    int64
NO_MUNICIPIO_PROVA      str
SG_UF_PROVA             str
RENDA_FAMILIAR          str
TIPO_DE_ESCOLA_EM       str
dtype: object

In [ ]:
df_resultados = pd.read_csv('./data/processed/RESULTADOS_TRATADOS.csv',sep=',', encoding='UTF-8')
df_resultados.head()

,NU_SEQUENCIAL,NU_ANO,CO_ESCOLA,CO_MUNICIPIO_ESC,NO_MUNICIPIO_ESC,CO_UF_ESC,SG_UF_ESC,TP_DEPENDENCIA_ADM_ESC,TP_LOCALIZACAO_ESC,TP_SIT_FUNC_ESC,NU_NOTA_CN,NU_NOTA_CH,NU_NOTA_LC,NU_NOTA_MT,NU_NOTA_REDACAO,MEDIA_GERAL
0,206403,2024,23052929,2301406,Aratuba,Ceará,CE,Estadual,Urbana,Em atividade,436.8,377.8,423.4,427.1,300.0,393.02
1,3604651,2024,42103770,4218004,Tijucas,Santa Catarina,SC,Privada,Urbana,Em atividade,521.9,601.9,605.5,689.2,920.0,667.70
2,1461268,2024,0,0,Não informado,Não informado,Não informado,Não informado,Não informado,Não informado,363.0,548.4,557.2,456.4,480.0,481.00
3,4301058,2024,0,0,Não informado,Não informado,Não informado,Não informado,Não informado,Não informado,550.7,553.8,605.9,629.1,740.0,615.90
4,3148322,2024,21150354,2100436,Alto Alegre do Maranhão,Maranhão,MA,Estadual,Urbana,Em atividade,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_resultados.dtypes

NU_SEQUENCIAL               int64
NU_ANO                      int64
CO_ESCOLA                   int64
CO_MUNICIPIO_ESC            int64
NO_MUNICIPIO_ESC              str
CO_UF_ESC                     str
SG_UF_ESC                     str
TP_DEPENDENCIA_ADM_ESC        str
TP_LOCALIZACAO_ESC            str
TP_SIT_FUNC_ESC               str
NU_NOTA_CN                float64
NU_NOTA_CH                float64
NU_NOTA_LC                float64
NU_NOTA_MT                float64
NU_NOTA_REDACAO           float64
MEDIA_GERAL               float64
dtype: object

In [ ]:
df_resultados = pd.read_csv('./data/processed/RESULTADOS_TRATADOS.csv',sep=',', encoding='UTF-8')
df_resultados.head()

,NU_SEQUENCIAL,NU_ANO,CO_ESCOLA,CO_MUNICIPIO_ESC,NO_MUNICIPIO_ESC,CO_UF_ESC,SG_UF_ESC,TP_DEPENDENCIA_ADM_ESC,TP_LOCALIZACAO_ESC,TP_SIT_FUNC_ESC,NU_NOTA_CN,NU_NOTA_CH,NU_NOTA_LC,NU_NOTA_MT,NU_NOTA_REDACAO,MEDIA_GERAL
0,206403,2024,23052929,2301406,Aratuba,Ceará,CE,Estadual,Urbana,Em atividade,436.8,377.8,423.4,427.1,300.0,393.02
1,3604651,2024,42103770,4218004,Tijucas,Santa Catarina,SC,Privada,Urbana,Em atividade,521.9,601.9,605.5,689.2,920.0,667.70
2,1461268,2024,0,0,Não informado,Não informado,Não informado,Não informado,Não informado,Não informado,363.0,548.4,557.2,456.4,480.0,481.00
3,4301058,2024,0,0,Não informado,Não informado,Não informado,Não informado,Não informado,Não informado,550.7,553.8,605.9,629.1,740.0,615.90
4,3148322,2024,21150354,2100436,Alto Alegre do Maranhão,Maranhão,MA,Estadual,Urbana,Em atividade,NaN,NaN,NaN,NaN,NaN,NaN


## VALIDAÇÃO MICRODADOS_ENEM_2024

In [ ]:
import chardet
with open("./data/processed/PARTICIPANTES_LIMPO.csv", "rb") as f:
    resultado = chardet.detect(f.read(100_000))
print(resultado)

{'encoding': 'utf-8', 'confidence': 0.8321252, 'language': 'pt', 'mime_type': 'text/plain'}


In [ ]:

df_teste = pd.read_csv("./data/processed/PARTICIPANTES_LIMPO.csv", sep=",", encoding="utf-8", nrows=5)
print(df_teste.columns.tolist())
print(df_teste.head())

['NU_INSCRICAO', 'NU_ANO', 'TP_FAIXA_ETARIA', 'TP_SEXO', 'TP_COR_RACA', 'CO_MUNICIPIO_PROVA', 'NO_MUNICIPIO_PROVA', 'SG_UF_PROVA', 'RENDA_FAMILIAR', 'TIPO_DE_ESCOLA_EM']
   NU_INSCRICAO  NU_ANO     TP_FAIXA_ETARIA TP_SEXO TP_COR_RACA  \
0  210062064233    2024             20 anos       F      Branca   
1  210062064234    2024  Entre 26 e 30 anos       F      Branca   
2  210062064235    2024  Entre 26 e 30 anos       F      Branca   
3  210062064236    2024             18 anos       F       Parda   
4  210062064237    2024  Entre 51 e 55 anos       M      Branca   

   CO_MUNICIPIO_PROVA NO_MUNICIPIO_PROVA SG_UF_PROVA RENDA_FAMILIAR  \
0             4314902       Porto Alegre          RS  DE 1 A 1.5 SM   
1             4318903   São Luiz Gonzaga          RS  DE 2 A 2.5 SM   
2             4320107            Sarandi          RS    DE 6 A 7 SM   
3             4313409      Novo Hamburgo          RS  DE 1 A 1.5 SM   
4             4309209           Gravataí          RS  NENHUMA RENDA   



In [ ]:
import chardet
with open("./data/processed/microdados_censo_2024_2025_tecnologia.csv", "rb") as f:
    resultado = chardet.detect(f.read(100_000))
print(resultado)

{'encoding': 'UTF-8-SIG', 'confidence': 1.0, 'language': 'pt', 'mime_type': 'text/plain'}


In [ ]:
df_teste = pd.read_csv("./data/processed/microdados_censo_2024_2025_tecnologia.csv", sep=";", encoding="utf-8", nrows=5)
print(df_teste.columns.tolist())
print(df_teste.head())

['CO_ESCOLA', 'COD_MUNICIPIO_IBGE', 'UF', 'REGIAO', 'NOME_ESCOLA', 'REDE_ENSINO', 'SITUACAO_FUNCIONAMENTO', 'TEM_INTERNET_ESCOLA', 'INTERNET_DISPONIVEL_ALUNOS', 'INTERNET_USO_PEDAGOGICO', 'TEM_BANDA_LARGA', 'TEM_LAB_INFORMATICA', 'QT_COMPUTADORES_DESKTOP', 'QT_NOTEBOOKS_ALUNO', 'QT_TABLETS_ALUNO', 'ANO_CENSO']
   CO_ESCOLA  COD_MUNICIPIO_IBGE  UF REGIAO                   NOME_ESCOLA  \
0   11022558             1100015  RO  Norte         EIEEF HAP BITT TUPARI   
1   11024275             1100015  RO  Norte      CEEJA LUIZ VAZ DE CAMOES   
2   11024291             1100015  RO  Norte           EMMEF 7 DE SETEMBRO   
3   11024666             1100015  RO  Norte          EMEIEF BOA ESPERANCA   
4   11024682             1100015  RO  Norte  EEEFM EURIDICE LOPES PEDROSO   

  REDE_ENSINO SITUACAO_FUNCIONAMENTO  TEM_INTERNET_ESCOLA  \
0    Estadual           Em Atividade                    1   
1    Estadual           Em Atividade                    1   
2   Municipal             Paralisada      

In [ ]:
df_resultados.columns

Index(['NU_SEQUENCIAL', 'NU_ANO', 'CO_ESCOLA', 'CO_MUNICIPIO_ESC',
       'NO_MUNICIPIO_ESC', 'CO_UF_ESC', 'SG_UF_ESC', 'TP_DEPENDENCIA_ADM_ESC',
       'TP_LOCALIZACAO_ESC', 'TP_SIT_FUNC_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH',
       'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'MEDIA_GERAL'],
      dtype='str')

In [ ]:
#transformando UF em formato de nome para COD
uf_map = {
    'Rondônia': 11,
    'Acre': 12,
    'Amazonas': 13,
    'Roraima': 14,
    'Pará': 15,
    'Amapá': 16,
    'Tocantins': 17,
    'Maranhão': 21,
    'Piauí': 22,
    'Ceará': 23,
    'Rio Grande do Norte': 24,
    'Paraíba': 25,
    'Pernambuco': 26,
    'Alagoas': 27,
    'Sergipe': 28,
    'Bahia': 29,
    'Minas Gerais': 31,
    'Espírito Santo': 32,
    'Rio de Janeiro': 33,
    'São Paulo': 35,
    'Paraná': 41,
    'Santa Catarina': 42,
    'Rio Grande do Sul': 43,
    'Mato Grosso do Sul': 50,
    'Mato Grosso': 51,
    'Goiás': 52,
    'Distrito Federal': 53
}

# 2. (Recomendado) Remover espaços em branco extras antes ou depois do nome
df_resultados['CO_UF_ESC'] = df_resultados['CO_UF_ESC'].astype(str).str.strip()

# 3. Mapear os valores do nome para o código correspondente
df_resultados['CO_UF_ESC'] = df_resultados['CO_UF_ESC'].map(uf_map)

# 4. Converter o tipo da coluna para inteiro
# Usa-se 'Int64' (com I maiúsculo) para que permita valores nulos (NaN) caso algum nome não seja encontrado
df_resultados['CO_UF_ESC'] = df_resultados['CO_UF_ESC'].astype('Int64')

In [ ]:
df_resultados

,NU_SEQUENCIAL,NU_ANO,CO_ESCOLA,CO_MUNICIPIO_ESC,NO_MUNICIPIO_ESC,CO_UF_ESC,SG_UF_ESC,TP_DEPENDENCIA_ADM_ESC,TP_LOCALIZACAO_ESC,TP_SIT_FUNC_ESC,NU_NOTA_CN,NU_NOTA_CH,NU_NOTA_LC,NU_NOTA_MT,NU_NOTA_REDACAO,MEDIA_GERAL
0,206403,2024,23052929,2301406,Aratuba,23,CE,Estadual,Urbana,Em atividade,436.8,377.8,423.4,427.1,300.0,393.02
1,3604651,2024,42103770,4218004,Tijucas,42,SC,Privada,Urbana,Em atividade,521.9,601.9,605.5,689.2,920.0,667.70
2,1461268,2024,0,0,Não informado,<NA>,Não informado,Não informado,Não informado,Não informado,363.0,548.4,557.2,456.4,480.0,481.00
3,4301058,2024,0,0,Não informado,<NA>,Não informado,Não informado,Não informado,Não informado,550.7,553.8,605.9,629.1,740.0,615.90
4,3148322,2024,21150354,2100436,Alto Alegre do Maranhão,21,MA,Estadual,Urbana,Em atividade,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9143711,1966623,2025,0,0,Não informado,<NA>,Não informado,Não informado,Não informado,Não informado,481.5,578.9,579.3,447.3,480.0,513.40
9143712,1661560,2025,29272696,2912301,Ibicuí,29,BA,Estadual,Urbana,Em atividade,NaN,NaN,NaN,NaN,NaN,NaN
9143713,2464303,2025,41133757,4106902,Curitiba,41,PR,Federal,Urbana,Em atividade,567.8,606.4,635.9,623.4,800.0,646.70
9143714,3164695,2025,0,0,Não informado,<NA>,Não informado,Não informado,Não informado,Não informado,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_resultados.duplicated().sum()

np.int64(2439230)

In [ ]:
df_resultados.to_csv('./data/processed/RESULTADOS_LIMPO.csv', index=False, sep=',', encoding='UTF-8')

In [ ]:
df_resultados.columns

Index(['NU_SEQUENCIAL', 'NU_ANO', 'CO_ESCOLA', 'CO_MUNICIPIO_ESC',
       'NO_MUNICIPIO_ESC', 'CO_UF_ESC', 'SG_UF_ESC', 'TP_DEPENDENCIA_ADM_ESC',
       'TP_LOCALIZACAO_ESC', 'TP_SIT_FUNC_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH',
       'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'MEDIA_GERAL'],
      dtype='str')

## Verificando duplicadas na coluna NU_SEQUENCIAL

In [ ]:

print(df_resultados["NU_SEQUENCIAL"].duplicated().sum())

4332944


In [ ]:
df_participantes= pd.read_csv('./data/processed/PARTICIPANTES_LIMPO.csv',sep=',', encoding='UTF-8')

In [ ]:
print(df_participantes["NU_INSCRICAO"].duplicated().sum())

0


## Excluindo coluna NU_SEQUENCIAL

In [ ]:
df_resultados = df_resultados.drop(columns=['NU_SEQUENCIAL'])

In [ ]:
df_resultados.to_csv('./data/processed/RESULTADOS_LIMPO_2.csv', index=False, sep=',', encoding='UTF-8')

In [ ]:
list(df_resultados.columns)

['NU_ANO',
 'CO_ESCOLA',
 'CO_MUNICIPIO_ESC',
 'NO_MUNICIPIO_ESC',
 'CO_UF_ESC',
 'SG_UF_ESC',
 'TP_DEPENDENCIA_ADM_ESC',
 'TP_LOCALIZACAO_ESC',
 'TP_SIT_FUNC_ESC',
 'NU_NOTA_CN',
 'NU_NOTA_CH',
 'NU_NOTA_LC',
 'NU_NOTA_MT',
 'NU_NOTA_REDACAO',
 'MEDIA_GERAL']

In [ ]:
df_resultados.dtypes

NU_ANO                      int64
CO_ESCOLA                   int64
CO_MUNICIPIO_ESC            int64
NO_MUNICIPIO_ESC              str
CO_UF_ESC                 float64
SG_UF_ESC                     str
TP_DEPENDENCIA_ADM_ESC        str
TP_LOCALIZACAO_ESC            str
TP_SIT_FUNC_ESC               str
NU_NOTA_CN                float64
NU_NOTA_CH                float64
NU_NOTA_LC                float64
NU_NOTA_MT                float64
NU_NOTA_REDACAO           float64
MEDIA_GERAL               float64
dtype: object

In [ ]:
df_censo = pd.read_csv('./data/processed/microdados_censo_2024_2025_tecnologia.csv',sep=';', encoding='UTF-8')
df_censo.head()

,CO_ESCOLA,COD_MUNICIPIO_IBGE,UF,REGIAO,NOME_ESCOLA,REDE_ENSINO,SITUACAO_FUNCIONAMENTO,TEM_INTERNET_ESCOLA,INTERNET_DISPONIVEL_ALUNOS,INTERNET_USO_PEDAGOGICO,TEM_BANDA_LARGA,TEM_LAB_INFORMATICA,QT_COMPUTADORES_DESKTOP,QT_NOTEBOOKS_ALUNO,QT_TABLETS_ALUNO,ANO_CENSO
0,11022558,1100015,RO,Norte,EIEEF HAP BITT TUPARI,Estadual,Em Atividade,1,0,0,0,0,0,1,0,2024
1,11024275,1100015,RO,Norte,CEEJA LUIZ VAZ DE CAMOES,Estadual,Em Atividade,1,1,1,1,1,19,4,0,2024
2,11024291,1100015,RO,Norte,EMMEF 7 DE SETEMBRO,Municipal,Paralisada,2,2,2,2,2,0,0,0,2024
3,11024666,1100015,RO,Norte,EMEIEF BOA ESPERANCA,Municipal,Em Atividade,1,0,0,0,0,0,0,0,2024
4,11024682,1100015,RO,Norte,EEEFM EURIDICE LOPES PEDROSO,Estadual,Em Atividade,1,1,1,1,1,19,0,0,2024


In [ ]:
df_censo.shape

(429737, 16)

In [ ]:
print("Quantidade de colunas:", len(df_censo.columns))
print(df_censo.columns.tolist())
display(df_censo.head())

Quantidade de colunas: 16
['CO_ESCOLA', 'COD_MUNICIPIO_IBGE', 'UF', 'REGIAO', 'NOME_ESCOLA', 'REDE_ENSINO', 'SITUACAO_FUNCIONAMENTO', 'TEM_INTERNET_ESCOLA', 'INTERNET_DISPONIVEL_ALUNOS', 'INTERNET_USO_PEDAGOGICO', 'TEM_BANDA_LARGA', 'TEM_LAB_INFORMATICA', 'QT_COMPUTADORES_DESKTOP', 'QT_NOTEBOOKS_ALUNO', 'QT_TABLETS_ALUNO', 'ANO_CENSO']


,CO_ESCOLA,COD_MUNICIPIO_IBGE,UF,REGIAO,NOME_ESCOLA,REDE_ENSINO,SITUACAO_FUNCIONAMENTO,TEM_INTERNET_ESCOLA,INTERNET_DISPONIVEL_ALUNOS,INTERNET_USO_PEDAGOGICO,TEM_BANDA_LARGA,TEM_LAB_INFORMATICA,QT_COMPUTADORES_DESKTOP,QT_NOTEBOOKS_ALUNO,QT_TABLETS_ALUNO,ANO_CENSO
0,11022558,1100015,RO,Norte,EIEEF HAP BITT TUPARI,Estadual,Em Atividade,1,0,0,0,0,0,1,0,2024
1,11024275,1100015,RO,Norte,CEEJA LUIZ VAZ DE CAMOES,Estadual,Em Atividade,1,1,1,1,1,19,4,0,2024
2,11024291,1100015,RO,Norte,EMMEF 7 DE SETEMBRO,Municipal,Paralisada,2,2,2,2,2,0,0,0,2024
3,11024666,1100015,RO,Norte,EMEIEF BOA ESPERANCA,Municipal,Em Atividade,1,0,0,0,0,0,0,0,2024
4,11024682,1100015,RO,Norte,EEEFM EURIDICE LOPES PEDROSO,Estadual,Em Atividade,1,1,1,1,1,19,0,0,2024


In [ ]:
df_censo.dtypes

CO_ESCOLA                     int64
COD_MUNICIPIO_IBGE            int64
UF                              str
REGIAO                          str
NOME_ESCOLA                     str
REDE_ENSINO                     str
SITUACAO_FUNCIONAMENTO          str
TEM_INTERNET_ESCOLA           int64
INTERNET_DISPONIVEL_ALUNOS    int64
INTERNET_USO_PEDAGOGICO       int64
TEM_BANDA_LARGA               int64
TEM_LAB_INFORMATICA           int64
QT_COMPUTADORES_DESKTOP       int64
QT_NOTEBOOKS_ALUNO            int64
QT_TABLETS_ALUNO              int64
ANO_CENSO                     int64
dtype: object

In [ ]:
df_censo.columns.to_list()

['CO_ESCOLA',
 'COD_MUNICIPIO_IBGE',
 'UF',
 'REGIAO',
 'NOME_ESCOLA',
 'REDE_ENSINO',
 'SITUACAO_FUNCIONAMENTO',
 'TEM_INTERNET_ESCOLA',
 'INTERNET_DISPONIVEL_ALUNOS',
 'INTERNET_USO_PEDAGOGICO',
 'TEM_BANDA_LARGA',
 'TEM_LAB_INFORMATICA',
 'QT_COMPUTADORES_DESKTOP',
 'QT_NOTEBOOKS_ALUNO',
 'QT_TABLETS_ALUNO',
 'ANO_CENSO']

In [ ]:
df_resultados.dtypes

NU_ANO                      int64
CO_ESCOLA                   int64
CO_MUNICIPIO_ESC            int64
NO_MUNICIPIO_ESC              str
CO_UF_ESC                 float64
SG_UF_ESC                     str
TP_DEPENDENCIA_ADM_ESC        str
TP_LOCALIZACAO_ESC            str
TP_SIT_FUNC_ESC               str
NU_NOTA_CN                float64
NU_NOTA_CH                float64
NU_NOTA_LC                float64
NU_NOTA_MT                float64
NU_NOTA_REDACAO           float64
MEDIA_GERAL               float64
dtype: object

In [ ]:
df_resultados

In [ ]:
df_ibge = pd.read_csv('./data/processed/ibge_agregado.csv',sep=',', encoding='UTF-8')
df_ibge.dtypes

COD_MUNICIPIO_IBGE         int64
POPULACAO_2022             int64
PIB_PER_CAPITA           float64
PIB_TOTAL_MIL_RS         float64
MUNICIPIO                    str
COD_UF_IBGE              float64
UF                           str
NOME_UF                      str
COD_REGIAO_IBGE          float64
REGIAO                       str
COD_MESORREGIAO_IBGE     float64
MESORREGIAO                  str
COD_MICRORREGIAO_IBGE    float64
MICRORREGIAO                 str
dtype: object

In [ ]:
df_ibge.isnull().sum()

COD_MUNICIPIO_IBGE       0
POPULACAO_2022           0
PIB_PER_CAPITA           0
PIB_TOTAL_MIL_RS         0
MUNICIPIO                0
COD_UF_IBGE              0
UF                       0
NOME_UF                  0
COD_REGIAO_IBGE          0
REGIAO                   0
COD_MESORREGIAO_IBGE     0
MESORREGIAO              0
COD_MICRORREGIAO_IBGE    0
MICRORREGIAO             0
dtype: int64

In [ ]:
df_censo.isnull().sum()

CO_ESCOLA                     0
COD_MUNICIPIO_IBGE            0
UF                            1
REGIAO                        1
NOME_ESCOLA                   0
REDE_ENSINO                   0
SITUACAO_FUNCIONAMENTO        0
TEM_INTERNET_ESCOLA           0
INTERNET_DISPONIVEL_ALUNOS    0
INTERNET_USO_PEDAGOGICO       0
TEM_BANDA_LARGA               0
TEM_LAB_INFORMATICA           0
QT_COMPUTADORES_DESKTOP       0
QT_NOTEBOOKS_ALUNO            0
QT_TABLETS_ALUNO              0
ANO_CENSO                     0
dtype: int64

In [ ]:
df_resultados.shape

(9143716, 16)

In [ ]:
df_resultados.head()

,NU_SEQUENCIAL,NU_ANO,CO_ESCOLA,CO_MUNICIPIO_ESC,NO_MUNICIPIO_ESC,CO_UF_ESC,SG_UF_ESC,TP_DEPENDENCIA_ADM_ESC,TP_LOCALIZACAO_ESC,TP_SIT_FUNC_ESC,NU_NOTA_CN,NU_NOTA_CH,NU_NOTA_LC,NU_NOTA_MT,NU_NOTA_REDACAO,MEDIA_GERAL
0,206403,2024,23052929,2301406,Aratuba,Ceará,CE,Estadual,Urbana,Em atividade,436.8,377.8,423.4,427.1,300.0,393.02
1,3604651,2024,42103770,4218004,Tijucas,Santa Catarina,SC,Privada,Urbana,Em atividade,521.9,601.9,605.5,689.2,920.0,667.70
2,1461268,2024,0,0,Não informado,Não informado,Não informado,Não informado,Não informado,Não informado,363.0,548.4,557.2,456.4,480.0,481.00
3,4301058,2024,0,0,Não informado,Não informado,Não informado,Não informado,Não informado,Não informado,550.7,553.8,605.9,629.1,740.0,615.90
4,3148322,2024,21150354,2100436,Alto Alegre do Maranhão,Maranhão,MA,Estadual,Urbana,Em atividade,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df = pd.read_csv('./data/processed/RESULTADOS_LIMPO_2.csv', sep=',')
df

,NU_ANO,CO_ESCOLA,CO_MUNICIPIO_ESC,NO_MUNICIPIO_ESC,CO_UF_ESC,SG_UF_ESC,TP_DEPENDENCIA_ADM_ESC,TP_LOCALIZACAO_ESC,TP_SIT_FUNC_ESC,NU_NOTA_CN,NU_NOTA_CH,NU_NOTA_LC,NU_NOTA_MT,NU_NOTA_REDACAO,MEDIA_GERAL
0,2024,23052929,2301406,Aratuba,23.0,CE,Estadual,Urbana,Em atividade,436.8,377.8,423.4,427.1,300.0,393.02
1,2024,42103770,4218004,Tijucas,42.0,SC,Privada,Urbana,Em atividade,521.9,601.9,605.5,689.2,920.0,667.70
2,2024,0,0,Não informado,NaN,Não informado,Não informado,Não informado,Não informado,363.0,548.4,557.2,456.4,480.0,481.00
3,2024,0,0,Não informado,NaN,Não informado,Não informado,Não informado,Não informado,550.7,553.8,605.9,629.1,740.0,615.90
4,2024,21150354,2100436,Alto Alegre do Maranhão,21.0,MA,Estadual,Urbana,Em atividade,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9143711,2025,0,0,Não informado,NaN,Não informado,Não informado,Não informado,Não informado,481.5,578.9,579.3,447.3,480.0,513.40
9143712,2025,29272696,2912301,Ibicuí,29.0,BA,Estadual,Urbana,Em atividade,NaN,NaN,NaN,NaN,NaN,NaN
9143713,2025,41133757,4106902,Curitiba,41.0,PR,Federal,Urbana,Em atividade,567.8,606.4,635.9,623.4,800.0,646.70
9143714,2025,0,0,Não informado,NaN,Não informado,Não informado,Não informado,Não informado,NaN,NaN,NaN,NaN,NaN,NaN
